# The generative model murder mystery

**IDETC-CIE 2026 · EngiBench hands-on workshop**

*Ten generative models were trained on the same topology optimization problem. You must put your detective hat on and find the best performing model. That is our killer.*

**Before you edit anything:** File → Save a copy in Drive. Opens read-only from GitHub.

---
## Setup

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "engibench[all] @ git+https://github.com/IDEALLab/EngiBench.git@main" "git+https://github.com/IDEALLab/EngiOpt.git@feat/idetc26-workshop"
    print("Installed. Runtime -> Restart session, then carry on from the next cell.")

In [ ]:
from engiopt.workshops.idetc26 import Case

case = Case.open("beams2d")     # <- the problem you are working on

The cheat sheet — refer to this anytime you want to call a function

In [ ]:
case.help()

---
# 1 · The scene of the crime (the dataset)

`"train"` — what the models were fitted on. `"test"` — the held-out set they are
scored against. Sliders move the conditions.

In [ ]:
case.show("train")

In [ ]:
case.problem.render(case.designs("test")[0])

---
# 2 · The suspects

The ten models in our lineup -- which one is best?

In [ ]:
case.models()

Each model has a case file explaining who they are. Pull it with `case.explain()`

In [ ]:
case.explain("knn_retrieval")

### The evidence (visualizing our suspects)

Show generated outputs from a given model

In [ ]:
case.show("diffusion", n=12)

Two suspects, same brief. Either side can be a model, `"test"`, or `"train"`.

In [ ]:
case.show("cgan_cnn_2d", "test")

`how=` swaps the picture:

| `how=` | draws |
|---|---|
| `"designs"` *(default)* | `n` designs from the suspect you name |
| `"compare"` | the suspects you name on the same briefs, real optimum on top |
| `"conditions"` | each design captioned `asked 0.30 / got 0.41` |
| `"nearest_training"` | each design above its closest training design |
| `"space_map"` | where a suspect's designs sit in a fitted space |

In [ ]:
case.show("knn_retrieval", "vqgan", how="compare", n=3)

---
# 3 · The interrogation/questions

We can look at many metrics to assist in our final verdict. Metrics provide insight into how well the model performs in various categories: cost, similarity, novelty, diversity, feasibility, and performance

In [ ]:
case.metrics()

## Cost — how expensive is the model?

If the training/inferencing the model is nearly as expensive as the traditional optimizer, is it worth it?

Cost metrics: `train_minutes`, `gen_seconds`, `params`

In [ ]:
case.evaluate("train_minutes").round(3)

## Similarity — does it look like the real thing?

How close do generated designs resemble traditional optimized designs?

Similarity metrics: `mmd`, `pixel_paired_distance`

In [ ]:
case.evaluate(["mmd", "pixel_paired_distance"]).round(4)

Who won? Every column in this family is maximized by handing back the training
data — and it is the family almost every paper reports, because it is the one
you can afford.

## Novelty — Are the designs new or memorized?

`novelty_ratio` is a scaled distance from each generated design to the nearest design
it could have copied. Around 1 is as novel as a real held-out design, near 0 is
memorized, far above 1 looks like nothing in the data — which random pixels also
achieve.

In [ ]:
case.evaluate("novelty_ratio").round(3)

By eye — each design above the closest thing to it in the training set.

In [ ]:
case.show("knn_retrieval", how="nearest_training")

## Diversity — How different are our generated designs from each other?

Diversity scores are hard to interpret without a relative scale. We can add some control data augmentations for reference with `controls=True`

| control | what it means |
|---|---|
| `collapsed` | an averaged design repeated 50 times |
| `noise_doped` | real optima plus Gaussian noise |
| `volume_only` | random blob that hits volume budget exactly|

Diversity metrics: `vendi`, `dpp`

In [ ]:
case.evaluate("pixel_vendi", controls=True).round(3)

In [ ]:
case.show("noise_doped")   # controls can be looked at like any other model

## Obedience — did it meet the constraints?

The design is not valid if it doesn't meet its budget

`cond_err` — Average distance from specified conditions (volume)
`viol` — a proportion of designs that missed by more than a tolerance

In [ ]:
case.evaluate(["cond_err", "viol"], controls=True).round(4)

In [ ]:
case.show("gan_cnn_2d", how="conditions")

## Performance — is the design actually any good?

Is the design close to optimum on generation (iog), does it require little effort to reach optimum on warmstart (cog), or does it converge to better/worse design when warmstarting (fog)?

In [ ]:
case.evaluate("cog", models=["knn_retrieval", "cgan_cnn_2d"]).round(3)

---
# 4 · The same questions, somewhere other than pixels

Every similarity/distance so far compared designs in pixel space. We can also project to featural spaces. Here we do so with PCA or through a learned autoencoder

| the question | pixels | PCA subspace | learned latent |
|---|---|---|---|
| does it look real? | `mmd` | `pca_mmd` | `lv_mmd` |
| how close to the right answer? | `pixel_paired_distance` | `pca_paired_distance` | `lv_paired_distance` |
| is it copying? | `novelty_ratio` | `pca_novelty` | `lv_novelty` |
| how many distinct designs? | `pixel_vendi` | `pca_vendi` | `lv_vendi` |
| did it cover the data? | — | `pca_coverage` | `lv_coverage` |

In [ ]:
case.show(case.evaluate(["mmd", "pca_mmd", "lv_mmd"]))

PCA and LV project to the same dimensionality for this notebook (determined through least volume). LV refers to the encoded latent space from `constrained_plvae_2d`

In [ ]:
case.latent_space()     # which autoencoder, how wide, and the PCA width matched to it

### Looking at the space

In [ ]:
case.show("knn_retrieval", "test", how="space_map")

Same designs, linear space. Nothing about the model changed; the picture can.

In [ ]:
case.show("knn_retrieval", "test", how="space_map", space="pca")

---
# 5 · Visualizing a board

In [ ]:
board = case.evaluate(["mmd", "pixel_vendi", "iog_median"])
case.show(board)

---
# 6 · Your accusation

Out loud, to the room:

1. **Who did it.** The model you think is the best.
2. **On what evidence.** Why do you think so?
3. **What you could not rule out.** What might have the decision easier?

The cell below is yours to build with.

Remember `case.help()`